# Inference
Вкажи шляхи нижче і запусти всі клітинки.

In [ ]:
# ── paths ────────────────────────────────────────────────────────────────
CKPT_PATHS = [
	'checkpoints/model_v5/fold1_epoch004_val0.6527.ckpt'
    # 'checkpoints/model_v1/fold1_epoch012_val0.6500.ckpt',  # додай решту фолдів для ансамблю
]
CSV_PATH    = '../data/test.csv'      # або train.csv
EEG_DIR     = '../data/test_eegs'    # або train_eegs
BACKBONE    = 'convnext_atto'     # має збігатись з тим що використовувалось під час навчання
BATCH_SIZE  = 32
NUM_WORKERS = 4
OUT_CSV     = 'submission.csv'


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path().resolve().parent))

from model.train import build_model
from model.dataset import EEGDataset, CLASS_NAMES, VOTE_COLS


In [3]:
device = (
    torch.device('mps')  if torch.backends.mps.is_available() else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)
print(f'device: {device}')


device: mps


In [6]:
df = pd.read_csv(CSV_PATH)

# один рядок на eeg_id — беремо перший sub_id
df_infer = df.groupby('eeg_id', as_index=False).first().reset_index(drop=True)

# якщо vote-колонок немає (test.csv) — заповнюємо рівномірно (не використовується при predict)
for col in VOTE_COLS:
    if col not in df_infer.columns:
        df_infer[col] = 1 / 6

dataset = EEGDataset(df_infer, EEG_DIR, augment=False)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True)
print(f'{len(dataset)} EEG samples')


1 EEG samples


In [7]:
all_fold_preds = []

for ckpt_path in CKPT_PATHS:
    model = build_model(BACKBONE, dropout=0.0).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    print(f'loaded  {Path(ckpt_path).name}')

    fold_preds = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc=Path(ckpt_path).stem, leave=False):
            imgs = imgs.to(device)
            probs = model(imgs).exp().cpu()
            fold_preds.append(probs)

    all_fold_preds.append(torch.cat(fold_preds))

# ансамбль: середнє по фолдах
preds = torch.stack(all_fold_preds).mean(dim=0).numpy()  # (N, 6)
print(f'preds: {preds.shape}')


FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/model_v1/fold0_epoch010_val0.6800.ckpt'

In [ ]:
sub = pd.DataFrame(preds, columns=VOTE_COLS)
sub.insert(0, 'eeg_id', df_infer['eeg_id'].values)

sub.to_csv(OUT_CSV, index=False)
print(f'saved → {OUT_CSV}')
sub.head(10)
